# Loss behaviour on MOJAVE images

One selected loss, or weighted loss combination, is evaluated under a controlled modification of a real MOJAVE crop. The response curve, modified image, residual, and optional gradient map describe the priorities of that objective.

All losses here act on images in the selected comparison space. Image MSE in this experiment is distinct from diffusion noise-prediction MSE. No model training or checkpoint is required.

## Initial defaults

These values initialize the interactive panel. Data locations come from `radio_diffusion.paths.get_mojave_data_dir` and the repository's local path configuration. Split selection is reproducible, and each modification retains an unchanged-image control.

The panel supports a single loss or a weighted combination. Effective combination weights are normalized automatically. Loss terms use the same selected image comparison space.

In [ ]:
IMAGE_SIZE = 256
SPLIT = "val"                 # "train", "val", or "test"
BATCH_SIZE = 4
BATCH_INDEX = 0
IMAGE_INDEX = 0               # Index within the selected batch.
SEED = 42
CPU_THREADS = 2              # Small image sweeps usually benefit from limited threading.

LOSS_NAMES = ["MSELoss"]      # Example combination: ["MSELoss", "SSIMLoss"]
LOSS_WEIGHTS = [1.0]          # Corresponding example weights: [0.7, 0.3]

NORMALIZATION = {
    "enabled": True,
    "transform": "linear",    # "linear", "log1p", or "signed_log1p"
    "bounds": "per_image",    # "per_image" or "shared"
    "log_scale": 1.0,
    "inverse_out_of_bounds": "linear",
}
COMPARISON_SPACE = "transformed"   # "physical" or "transformed"

MODIFICATION = "blur"
STRENGTHS = None              # None uses the selected modification's preset.
PREVIEW_STRENGTH = None       # None selects the middle of the preset interval.
N_STEPS = 21

MODIFICATION_OPTIONS = {
    "shift_angle_deg": 0.0,
    "fill_value": 0.0,
    "core_center_xy": None,   # None locates the reference's brightest pixel.
    "core_radius_px": 8.0,
    "core_edge_px": 1.5,
    "jet_min_sigma": 3.0,
    "jet_max_peak_fraction": 0.3,
    "artifact_center_fraction": (0.75, 0.25),
    "artifact_sigma_px": 3.0,
}

DISPLAY_STRETCH = "asinh"     # Display only: "linear" or "asinh".
SHOW_GRADIENT = True
USE_WIDGETS = True

## Imports and project setup

The package bootstrap supports execution from the repository root or its notebooks directory. Dataset locations are resolved only through the project's path helpers.

In [ ]:
import math
import contextlib
import io
from html import escape
import sys
from functools import partial
from itertools import islice
from pathlib import Path

# Locate the package if the current kernel does not have an editable installation.
try:
    from radio_diffusion.paths import REPOSITORY_ROOT, get_mojave_data_dir
except ModuleNotFoundError as error:
    if error.name != "radio_diffusion":
        raise
    package_root = next(
        (root for root in (Path.cwd(), *Path.cwd().parents)
         if (root / "src" / "radio_diffusion" / "paths.py").is_file()),
        None,
    )
    if package_root is None:
        raise RuntimeError("The radio_diffusion package or repository root could not be found.") from error
    sys.path.insert(0, str(package_root / "src"))
    from radio_diffusion.paths import REPOSITORY_ROOT, get_mojave_data_dir

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
if not isinstance(CPU_THREADS, int) or CPU_THREADS < 1:
    raise ValueError("CPU_THREADS must be a positive integer.")
torch.set_num_threads(CPU_THREADS)
import torch.nn.functional as F
from astropy.visualization import AsinhStretch, ImageNormalize
from IPython import get_ipython
from IPython.display import display, clear_output
shell = get_ipython()
if shell is not None and getattr(shell, "kernel", None) is not None:
    shell.run_line_magic("matplotlib", "inline")
from matplotlib.colors import Normalize, TwoSlopeNorm
from torch.utils.data import DataLoader
from torchvision.transforms.functional import gaussian_blur

from radio_diffusion.data.mojave import MojaveDataModule, MojaveDataset
from radio_diffusion.load_config import NormalizationSettings
from radio_diffusion.normalization import ImageIntensityTransform
from radio_diffusion.training.losses import WeightedLoss

## Interactive experiment panel

An initial top-to-bottom execution initializes this panel. Loss, modification, normalization, image selection, and display settings are then changed through the controls. Preview updates occur after slider release. Full response curves are calculated with **Run sweep**; cached terms are reused when only losses or weights change.

In [ ]:
widgets = None
if USE_WIDGETS:
    try:
        import ipywidgets as widgets
    except ImportError:
        print("ipywidgets is unavailable; the static experiment remains available below.")
WIDGETS_AVAILABLE = widgets is not None
if WIDGETS_AVAILABLE:
    dashboard_mount = widgets.VBox([widgets.HTML("<i>Initialising experiment controls…</i>")])
    display(dashboard_mount)

## Dataloader

The data module supplies its existing split and, when needed, shared normalization bounds. A raw `MojaveDataset` view of that split supplies physical crops and metadata through a dataloader. Existing processed data is read directly; preprocessing is not invoked.

In [ ]:
def make_loader(image_size, split, batch_size, seed, normalization_settings):
    if split not in {"train", "val", "test"}:
        raise ValueError("SPLIT must be train, val, or test.")
    if not isinstance(batch_size, int) or batch_size < 1:
        raise ValueError("BATCH_SIZE must be a positive integer.")
    data_dir = get_mojave_data_dir(image_size)
    data = MojaveDataModule(
        data_dir=data_dir,
        expected_image_shape=(1, image_size, image_size),
        image_size=image_size,
        batch_size=batch_size,
        num_workers=0,
        seed=seed,
        normalization=normalization_settings,
    )
    data.setup("test" if split == "test" else "fit")
    split_dataset = getattr(data, f"{split}_dataset")
    if len(split_dataset) == 0:
        raise ValueError(f"The {split} split is empty; select another split or dataset.")
    raw_dataset = MojaveDataset(
        split_dataset.paths,
        expected_image_shape=(1, image_size, image_size),
        normalization=NormalizationSettings(enabled=False),
        metadata_fields=[
            "source_name", "noise_rms_jy_per_beam",
            "cropped_image_pixel_minimum", "cropped_image_pixel_maximum",
        ],
    )
    loader = DataLoader(raw_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    return data, loader

In [ ]:
def select_reference(loader, batch_index, image_index):
    if not isinstance(batch_index, int) or batch_index < 0:
        raise ValueError("BATCH_INDEX must be a nonnegative integer.")
    batch = next(islice(iter(loader), batch_index, None), None)
    if batch is None:
        raise IndexError(f"Batch {batch_index} does not exist; the loader contains {len(loader)} batches.")
    if not isinstance(image_index, int) or not 0 <= image_index < len(batch["image"]):
        raise IndexError(f"IMAGE_INDEX must be between 0 and {len(batch['image']) - 1}.")
    image = batch["image"][image_index].clone().float()
    if not torch.isfinite(image).all():
        raise ValueError("The reference crop contains nonfinite pixels.")
    metadata = {}
    validity = {}
    for name, values in batch["metadata"].items():
        value = values[image_index]
        metadata[name] = value.item() if isinstance(value, torch.Tensor) and value.ndim == 0 else value
        validity[name] = bool(batch["metadata_mask"][name][image_index])
    path_index = batch_index * loader.batch_size + image_index
    path = Path(loader.dataset.paths[path_index])
    return image, metadata, validity, path

## Fixed reference context

Normalization bounds, the core/jet masks, and any random noise realization are established once from the reference. Invalid or missing saved noise RMS uses a border-MAD estimate, with a peak-relative fallback for constant backgrounds. The masks are illustrative image-space regions, not fitted physical jet components.

In [ ]:
def make_reference_context(reference, metadata, validity, shared_bounds, settings, options, seed):
    height, width = reference.shape[-2:]
    minimum = float(reference.min())
    maximum = float(reference.max())
    bounds = {
        "cropped_image_pixel_minimum": minimum,
        "cropped_image_pixel_maximum": maximum,
        **shared_bounds,
    }
    for name in ("cropped_image_pixel_minimum", "cropped_image_pixel_maximum"):
        if validity.get(name, False):
            bounds[name] = float(metadata[name])
    if bounds["cropped_image_pixel_minimum"] > bounds["cropped_image_pixel_maximum"]:
        raise ValueError("Saved crop bounds are reversed.")

    noise = float(metadata.get("noise_rms_jy_per_beam", 0.0))
    noise_source = "saved physical RMS"
    if not validity.get("noise_rms_jy_per_beam", False) or not math.isfinite(noise) or noise <= 0:
        border_width = max(1, min(height, width) // 8)
        border = torch.cat([
            reference[:, :border_width].flatten(), reference[:, -border_width:].flatten(),
            reference[:, border_width:-border_width, :border_width].flatten(),
            reference[:, border_width:-border_width, -border_width:].flatten(),
        ])
        noise = float((border - border.median()).abs().median() * 1.4826)
        noise_source = "border MAD estimate"
        if not math.isfinite(noise) or noise <= 0:
            noise = max(abs(minimum), abs(maximum), 1e-12) * 1e-3
            noise_source = "peak-relative fallback"

    center = options["core_center_xy"]
    if center is None:
        index = int(reference[0].argmax())
        center = (index % width, index // width)
    center_x, center_y = map(float, center)
    if not (0 <= center_x < width and 0 <= center_y < height):
        raise ValueError("The selected core center lies outside the crop.")
    if options["core_radius_px"] <= 0 or options["core_edge_px"] <= 0:
        raise ValueError("Core radius and edge width must be positive.")
    yy, xx = torch.meshgrid(
        torch.arange(height, dtype=reference.dtype),
        torch.arange(width, dtype=reference.dtype), indexing="ij",
    )
    radius = ((xx - center_x).square() + (yy - center_y).square()).sqrt()
    core_mask = torch.sigmoid((options["core_radius_px"] - radius) / options["core_edge_px"])
    jet_mask = (
        (reference[0] > options["jet_min_sigma"] * noise)
        & (reference[0] < options["jet_max_peak_fraction"] * max(maximum, 0.0))
    ).to(reference.dtype) * (1 - core_mask)
    generator = torch.Generator().manual_seed(seed)
    fixed_noise = torch.randn(reference.shape, dtype=reference.dtype, generator=generator)
    return {
        "bounds": bounds,
        "noise_rms": noise,
        "noise_source": noise_source,
        "peak": max(maximum, noise),
        "core_center_xy": (center_x, center_y),
        "core_mask": core_mask.unsqueeze(0),
        "jet_mask": jet_mask.unsqueeze(0),
        "fixed_noise": fixed_noise,
        "options": dict(options),
    }

In [ ]:
normalization_settings = NormalizationSettings(**NORMALIZATION)
intensity_transform = ImageIntensityTransform(normalization_settings)
loss_function = WeightedLoss(LOSS_NAMES, LOSS_WEIGHTS)
if COMPARISON_SPACE not in {"physical", "transformed"}:
    raise ValueError("COMPARISON_SPACE must be physical or transformed.")
if DISPLAY_STRETCH not in {"linear", "asinh"}:
    raise ValueError("DISPLAY_STRETCH must be linear or asinh.")
if not isinstance(N_STEPS, int) or N_STEPS < 2:
    raise ValueError("N_STEPS must be an integer of at least two.")

data, loader = make_loader(IMAGE_SIZE, SPLIT, BATCH_SIZE, SEED, normalization_settings)
reference, reference_metadata, metadata_validity, reference_path = select_reference(
    loader, BATCH_INDEX, IMAGE_INDEX,
)
context = make_reference_context(
    reference, reference_metadata, metadata_validity, data.dataset_metadata,
    normalization_settings, MODIFICATION_OPTIONS, SEED,
)

if not WIDGETS_AVAILABLE:
    print(f"Dataset: {data.data_dir}; reference: {reference_path.name}")
    display(pd.DataFrame({"value": reference_metadata, "valid": metadata_validity}))

## Comparison-space preparation

Physical comparisons use the original cropped brightness directly. Transformed comparisons apply the same configured intensity transform and fixed reference bounds to both images. The inverse policy is recorded in the settings but is not invoked in this physical-image perturbation experiment.

SSIM uses range one after enabled normalization. Otherwise its range comes from the fixed physical bounds; a constant reference uses range one, matching the training integration. No candidate image is independently rescaled.

In [ ]:
def prepare_comparison(reference, modified, mapping, context, space):
    if modified.shape != reference.shape or not modified.is_floating_point():
        raise ValueError("A modification must return a floating-point tensor with the reference shape.")
    if modified.device != reference.device or not torch.isfinite(modified).all():
        raise ValueError("A modification must return finite pixels on the reference device.")
    settings = mapping.settings
    if space == "transformed" and settings.enabled:
        target = mapping.normalize(reference, context["bounds"])
        prediction = mapping.normalize(modified, context["bounds"])
        data_range = 1.0
    elif space in {"physical", "transformed"}:
        target, prediction = reference, modified
        prefix = "dataset_pixel" if settings.enabled and settings.bounds == "shared" else "cropped_image_pixel"
        data_range = context["bounds"][f"{prefix}_maximum"] - context["bounds"][f"{prefix}_minimum"]
        if not math.isfinite(data_range) or data_range < 0:
            raise ValueError("The fixed physical comparison range must be finite and nonnegative.")
        data_range = data_range if data_range > 0 else 1.0
    else:
        raise ValueError("Comparison space must be physical or transformed.")
    return prediction.unsqueeze(0), target.unsqueeze(0), data_range

## Modification functions

Each function returns a new physical image. Zero strength is the unchanged-image control. The shared context contains only reference-derived quantities; changing a candidate does not update its masks, bounds, or noise pattern.

### Blur

Strength is Gaussian sigma in pixels. Reflective padding defines the crop-edge behaviour; broadening near a boundary need not preserve the aperture brightness sum.

In [ ]:
def blur_image(image, strength, context):
    sigma = float(strength)
    if not math.isfinite(sigma) or sigma < 0:
        raise ValueError("Blur sigma must be finite and nonnegative.")
    if sigma == 0:
        return image.clone()
    radius = max(1, math.ceil(3 * sigma))
    if radius >= min(image.shape[-2:]):
        raise ValueError("Blur radius is too large for reflective padding of this crop.")
    return gaussian_blur(image, kernel_size=2 * radius + 1, sigma=sigma)

### Translation

Strength is displacement in pixels along the configured angle. Bilinear sampling and a fixed physical fill value avoid periodic wraparound.

In [ ]:
def shift_image(image, strength, context):
    distance = float(strength)
    if not math.isfinite(distance):
        raise ValueError("Shift distance must be finite.")
    if distance == 0:
        return image.clone()
    angle = math.radians(context["options"]["shift_angle_deg"])
    height, width = image.shape[-2:]
    theta = image.new_tensor([
        [1.0, 0.0, -2 * distance * math.cos(angle) / width],
        [0.0, 1.0, -2 * distance * math.sin(angle) / height],
    ]).unsqueeze(0)
    grid = F.affine_grid(theta, (1, *image.shape), align_corners=False)
    fill = float(context["options"]["fill_value"])
    return F.grid_sample(
        (image - fill).unsqueeze(0), grid,
        mode="bilinear", padding_mode="zeros", align_corners=False,
    )[0] + fill

### Core displacement

The reference-derived soft core component is translated while the remaining image is retained. This is a controlled image-space modification, not a fitted component-motion model.

In [ ]:
def shift_core(image, strength, context):
    if float(strength) == 0:
        return image.clone()
    core = image * context["core_mask"]
    return image - core + shift_image(core, strength, {
        **context, "options": {**context["options"], "fill_value": 0.0},
    })

### Brightness gain

Strength is the relative gain change: 0.2 multiplies brightness by 1.2. Fixed normalization bounds retain this change in the loss input.

In [ ]:
def change_brightness(image, strength, context):
    change = float(strength)
    if not math.isfinite(change) or change < -1:
        raise ValueError("Relative brightness gain must be finite and at least -1.")
    return image * (1 + change)

### Background offset

Strength is an additive offset in units of the reference's physical noise RMS.

In [ ]:
def offset_background(image, strength, context):
    offset = float(strength)
    if not math.isfinite(offset):
        raise ValueError("Background offset must be finite.")
    return image + offset * context["noise_rms"]

### Faint-jet attenuation

Strength ranges from zero to one. Positive brightness is attenuated in the fixed reference mask outside the core, above the selected noise threshold and below the selected fraction of peak brightness. An empty mask is reported rather than presented as a meaningful response.

In [ ]:
def attenuate_jet(image, strength, context):
    fraction = float(strength)
    if not math.isfinite(fraction) or not 0 <= fraction <= 1:
        raise ValueError("Jet attenuation must lie between zero and one.")
    if fraction == 0:
        return image.clone()
    if not torch.any(context["jet_mask"] > 0):
        raise ValueError("The reference jet mask is empty; adjust the mask settings or select another crop.")
    return image - fraction * context["jet_mask"] * image.clamp_min(0)

### Spurious component

Strength is the added Gaussian component's peak amplitude as a fraction of the reference peak. Its position and width remain fixed throughout the sweep.

In [ ]:
def add_artifact(image, strength, context):
    amplitude = float(strength)
    if not math.isfinite(amplitude) or amplitude < 0:
        raise ValueError("Artifact amplitude must be finite and nonnegative.")
    sigma = float(context["options"]["artifact_sigma_px"])
    if sigma <= 0 or not math.isfinite(sigma):
        raise ValueError("Artifact width must be finite and positive.")
    height, width = image.shape[-2:]
    center_x, center_y = context["options"]["artifact_center_fraction"]
    if not 0 <= center_x <= 1 or not 0 <= center_y <= 1:
        raise ValueError("Artifact center fractions must lie between zero and one.")
    yy, xx = torch.meshgrid(
        torch.arange(height, dtype=image.dtype, device=image.device),
        torch.arange(width, dtype=image.dtype, device=image.device), indexing="ij",
    )
    component = torch.exp(-(
        (xx - center_x * (width - 1)).square() + (yy - center_y * (height - 1)).square()
    ) / (2 * sigma**2))
    return image + amplitude * context["peak"] * component.unsqueeze(0)

### Added noise

Strength is the added noise standard deviation divided by the reference RMS. A single seeded noise realization is scaled throughout the sweep.

In [ ]:
def add_noise(image, strength, context):
    amplitude = float(strength)
    if not math.isfinite(amplitude) or amplitude < 0:
        raise ValueError("Added-noise amplitude must be finite and nonnegative.")
    return image + amplitude * context["noise_rms"] * context["fixed_noise"]

## Modification registry

A new modification can be added as its own function cell and one registry entry with a strength interval and axis label.

In [ ]:
MODIFICATIONS = {
    "blur": {"function": blur_image, "limits": (0.0, 4.0), "label": "Gaussian sigma [pixels]"},
    "shift": {"function": shift_image, "limits": (0.0, 8.0), "label": "Image displacement [pixels]"},
    "core_shift": {"function": shift_core, "limits": (0.0, 8.0), "label": "Core displacement [pixels]"},
    "brightness": {"function": change_brightness, "limits": (-0.5, 1.0), "label": "Relative brightness gain"},
    "offset": {"function": offset_background, "limits": (-3.0, 3.0), "label": "Background offset / reference RMS"},
    "jet_attenuation": {"function": attenuate_jet, "limits": (0.0, 1.0), "label": "Fraction of selected jet emission removed"},
    "artifact": {"function": add_artifact, "limits": (0.0, 0.25), "label": "Artifact peak / reference peak"},
    "noise": {"function": add_noise, "limits": (0.0, 5.0), "label": "Added noise sigma / reference RMS"},
}
if MODIFICATION not in MODIFICATIONS:
    raise ValueError(f"Unknown modification {MODIFICATION!r}; expected one of {list(MODIFICATIONS)}.")
selected_modification = MODIFICATIONS[MODIFICATION]
strengths = np.asarray(
    STRENGTHS if STRENGTHS is not None else np.linspace(*selected_modification["limits"], N_STEPS),
    dtype=float,
)
if strengths.ndim != 1 or strengths.size == 0 or not np.isfinite(strengths).all():
    raise ValueError("STRENGTHS must be a nonempty, finite one-dimensional sequence.")
strengths = np.unique(np.r_[0.0, strengths])
preview_strength = (
    float(PREVIEW_STRENGTH) if PREVIEW_STRENGTH is not None
    else float((strengths.min() + strengths.max()) / 2)
)

## Loss evaluation and severity sweep

The project loss object supplies the scalar total and individual terms. Weighted contributions describe the selected combination. Invalid points, such as a logarithm outside its domain, retain an error message and appear as gaps in the response curve.

In [ ]:
def evaluate_loss(reference, modified, criterion, mapping, context, space):
    prediction, target, data_range = prepare_comparison(reference, modified, mapping, context, space)
    image_inputs = {}
    if criterion.requires_images:
        image_inputs = {
            "image_prediction": prediction,
            "image_target": target,
            "ssim_data_range": data_range,
        }
    total, terms = criterion.compute(prediction, target, **image_inputs)
    if not torch.isfinite(total) or any(not torch.isfinite(term) for term in terms.values()):
        raise FloatingPointError("The selected loss produced a nonfinite result.")
    return total, terms

In [ ]:
def run_sweep(reference, modification, strengths, criterion, mapping, context, space):
    rows = []
    for strength in strengths:
        row = {"strength": float(strength), "total": np.nan, "error": ""}
        try:
            with torch.no_grad():
                modified = modification(reference, float(strength), context)
                total, terms = evaluate_loss(reference, modified, criterion, mapping, context, space)
            row.update({
                "total": float(total),
                "modified_min": float(modified.min()),
                "modified_max": float(modified.max()),
                "outside_reference_fraction": float((
                    (modified < context["bounds"]["cropped_image_pixel_minimum"])
                    | (modified > context["bounds"]["cropped_image_pixel_maximum"])
                ).float().mean()),
            })
            for name, weight in zip(criterion.loss_names, criterion.weights):
                row[f"term:{name}"] = float(terms[name])
                row[f"weighted:{name}"] = weight * float(terms[name])
        except (ValueError, RuntimeError, FloatingPointError, OverflowError) as error:
            row["error"] = f"{type(error).__name__}: {error}"
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
def loss_gradient(reference, modified, criterion, mapping, context, space):
    candidate = modified.detach().clone().requires_grad_(True)
    total, _ = evaluate_loss(reference, candidate, criterion, mapping, context, space)
    gradient, = torch.autograd.grad(total, candidate)
    if not torch.isfinite(gradient).all():
        raise FloatingPointError("The selected loss produced a nonfinite pixel gradient.")
    return gradient.detach()

## Plotting functions

Reference and modified images share a reference-derived display scale. Display stretch does not change loss inputs. Saturation beyond those limits is visible through the reported pixel ranges; the loss receives the actual values. Residual limits also remain tied to the reference span.

In [ ]:
def plot_response(results, criterion, axis_label, preview_strength=None, preview_value=None):
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    ax.plot(results["strength"], results["total"], "o-", markersize=3, label="Selected objective")
    if len(criterion.loss_names) > 1:
        for name in criterion.loss_names:
            ax.plot(results["strength"], results.get(f"weighted:{name}", np.nan), "--", label=f"Weighted {name}")
    if preview_strength is not None:
        ax.axvline(preview_strength, color="0.5", linewidth=1, linestyle=":")
    if preview_value is not None and math.isfinite(preview_value):
        ax.scatter([preview_strength], [preview_value], color="black", zorder=4, label="Preview")
    ax.set(xlabel=axis_label, ylabel="Loss", title=f"{dict(zip(criterion.loss_names, criterion.weights))}")
    ax.grid(alpha=0.25)
    ax.legend()
    return fig

In [ ]:
def plot_image_pair(reference, modified, title, unit_label, stretch="asinh"):
    reference_array = reference.detach().cpu()[0].numpy()
    modified_array = modified.detach().cpu()[0].numpy()
    minimum, maximum = float(reference_array.min()), float(reference_array.max())
    span = maximum - minimum
    if span == 0:
        span = max(abs(minimum), 1.0)
        maximum = minimum + span
    image_norm = (
        ImageNormalize(vmin=minimum, vmax=maximum, stretch=AsinhStretch(0.03), clip=False)
        if stretch == "asinh" else Normalize(vmin=minimum, vmax=maximum)
    )
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), constrained_layout=True)
    arrays = [reference_array, modified_array, modified_array - reference_array]
    titles = ["Reference", "Modified", "Modified - reference"]
    for index, (ax, array, label) in enumerate(zip(axes, arrays, titles)):
        norm = image_norm if index < 2 else TwoSlopeNorm(vmin=-span, vcenter=0, vmax=span)
        artist = ax.imshow(array, origin="lower", cmap="inferno" if index < 2 else "RdBu_r", norm=norm)
        ax.set(title=f"{label}\nrange [{array.min():.3g}, {array.max():.3g}]", xlabel="x [pixels]", ylabel="y [pixels]")
        fig.colorbar(artist, ax=ax, shrink=0.8, label=unit_label, extend="both")
    fig.suptitle(title)
    return fig

In [ ]:
def plot_gradient(gradient, title):
    values = gradient.detach().cpu()[0].numpy()
    limit = float(np.abs(values).max())
    limit = limit if limit > 0 else 1.0
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    magnitude = axes[0].imshow(np.abs(values), origin="lower", cmap="magma", vmin=0, vmax=limit)
    direction = axes[1].imshow(-values, origin="lower", cmap="RdBu_r", vmin=-limit, vmax=limit)
    axes[0].set_title("Sensitivity magnitude")
    axes[1].set_title("Negative gradient: local descent direction")
    for ax in axes:
        ax.set(xlabel="x [pixels]", ylabel="y [pixels]")
    fig.colorbar(magnitude, ax=axes[0], label="Absolute loss gradient")
    fig.colorbar(direction, ax=axes[1], label="Negative loss gradient")
    fig.suptitle(title + " — gradient with respect to physical modified pixels")
    return fig

In [ ]:
def show_preview(strength, reference, modification, criterion, mapping, context, space, results, axis_label, title, stretch, show_gradient):
    modified = modification(reference, float(strength), context)
    total_value = None
    try:
        total, terms = evaluate_loss(reference, modified, criterion, mapping, context, space)
        total_value = float(total.detach())
        print(f"Strength: {strength:.6g}; selected loss: {total_value:.6g}; comparison space: {space}")
        display(pd.DataFrame([
            {"term": name, "weight": weight, "unweighted": float(terms[name].detach()),
             "weighted": weight * float(terms[name].detach())}
            for name, weight in zip(criterion.loss_names, criterion.weights)
        ]))
    except (ValueError, RuntimeError, FloatingPointError, OverflowError) as error:
        print(f"Invalid comparison at this strength: {error}")
    if results is not None:
        fig = plot_response(results, criterion, axis_label, float(strength), total_value)
        display(fig)
        plt.close(fig)
    fig = plot_image_pair(reference, modified, title + " — physical crop", "Jy/beam", stretch)
    display(fig)
    plt.close(fig)
    if total_value is not None and space == "transformed" and mapping.settings.enabled:
        prediction, target, _ = prepare_comparison(reference, modified, mapping, context, space)
        fig = plot_image_pair(target[0], prediction[0], title + " — actual loss inputs", "Transformed intensity", stretch)
        display(fig)
        plt.close(fig)
    if total_value is not None and show_gradient:
        gradient = loss_gradient(reference, modified, criterion, mapping, context, space)
        fig = plot_gradient(gradient, title)
        display(fig)
        plt.close(fig)

## Dashboard helpers

Loaded references, their fixed contexts, and evaluated loss terms are cached separately. Weight changes recombine existing terms without reevaluating the sweep. A new image, modification, normalization, or strength grid requires a new sweep. Cache entries are bounded to limit memory use.

In [ ]:
def dashboard_loader(state, split):
    if split not in state["loaders"]:
        with contextlib.redirect_stdout(io.StringIO()):
            state["loaders"][split] = make_loader(
                IMAGE_SIZE, split, BATCH_SIZE, SEED, NormalizationSettings(enabled=False),
            )
    return state["loaders"][split]

In [ ]:
def dashboard_reference(state, split, index):
    key = (split, int(index))
    if key not in state["images"]:
        _, current_loader = dashboard_loader(state, split)
        state["images"][key] = select_reference(
            current_loader, int(index) // BATCH_SIZE, int(index) % BATCH_SIZE,
        )
        if len(state["images"]) > 8:
            state["images"].pop(next(iter(state["images"])))
    return state["images"][key]

In [ ]:
def selected_dashboard_loss(controls):
    if controls["mode"].value == "single":
        return [controls["single_loss"].value], [1.0]
    names = [name for name, widget in controls["loss_checks"].items() if widget.value]
    raw_weights = [controls["loss_weights"][name].value for name in names]
    total = math.fsum(raw_weights)
    if not names or total <= 0:
        raise ValueError("Select at least one loss with a positive weight.")
    return names, [weight / total for weight in raw_weights]

In [ ]:
def dashboard_experiment(controls, state):
    names, weights = selected_dashboard_loss(controls)
    settings = NormalizationSettings(
        enabled=controls["normalize"].value, transform=controls["transform"].value,
        bounds=controls["bounds"].value, log_scale=controls["log_scale"].value,
        inverse_out_of_bounds=NORMALIZATION.get("inverse_out_of_bounds") or "linear",
    )
    split, index = controls["split"].value, controls["image"].value
    reference_key = (split, index)
    current_reference, metadata, validity, path = dashboard_reference(state, split, index)
    if settings.enabled and settings.bounds == "shared" and not state["shared_bounds"]:
        with contextlib.redirect_stdout(io.StringIO()):
            shared_data, _ = make_loader(IMAGE_SIZE, split, BATCH_SIZE, SEED, settings)
        state["shared_bounds"] = dict(shared_data.dataset_metadata)
    options = {
        **MODIFICATION_OPTIONS,
        "shift_angle_deg": controls["angle"].value, "fill_value": controls["fill"].value,
        "core_radius_px": controls["core_radius"].value, "core_edge_px": controls["core_edge"].value,
        "jet_min_sigma": controls["jet_sigma"].value, "jet_max_peak_fraction": controls["jet_fraction"].value,
        "artifact_center_fraction": (controls["artifact_x"].value, controls["artifact_y"].value),
        "artifact_sigma_px": controls["artifact_width"].value,
    }
    context_key = (
        reference_key, tuple(sorted(options.items())), controls["noise_seed"].value,
        tuple(sorted(state["shared_bounds"].items())),
    )
    if context_key not in state["contexts"]:
        state["contexts"][context_key] = make_reference_context(
            current_reference, metadata, validity, state["shared_bounds"],
            settings, options, controls["noise_seed"].value,
        )
        if len(state["contexts"]) > 8:
            state["contexts"].pop(next(iter(state["contexts"])))
    lower, upper = controls["range_min"].value, controls["range_max"].value
    if not math.isfinite(lower) or not math.isfinite(upper) or lower >= upper:
        raise ValueError("Sweep minimum must be finite and smaller than its maximum.")
    grid = np.unique(np.r_[0.0, np.linspace(lower, upper, controls["steps"].value)])
    modification_name = controls["modification"].value
    specification = MODIFICATIONS[modification_name]
    curve_key = (
        context_key, modification_name, id(specification["function"]),
        settings.enabled, settings.transform, settings.bounds, settings.log_scale,
        controls["space"].value, tuple(float(value) for value in grid),
    )
    return {
        "reference": current_reference, "reference_key": reference_key, "metadata": metadata,
        "path": path, "context": state["contexts"][context_key],
        "mapping": ImageIntensityTransform(settings), "criterion": WeightedLoss(names, weights),
        "space": controls["space"].value, "modification": specification["function"],
        "name": modification_name, "label": specification["label"], "grid": grid,
        "curve_key": curve_key, "strength": controls["strength"].value,
    }

In [ ]:
def cached_dashboard_sweep(experiment, state, run_missing=False):
    key = experiment["curve_key"]
    if key not in state["curves"]:
        if len(state["curves"]) >= 16:
            state["curves"].pop(next(iter(state["curves"])))
        state["curves"][key] = {}
    cache = state["curves"][key]
    criterion = experiment["criterion"]
    missing = [name for name in criterion.loss_names if name not in cache]
    if run_missing:
        for name in missing:
            cache[name] = run_sweep(
                experiment["reference"], experiment["modification"], experiment["grid"],
                WeightedLoss([name], [1.0]), experiment["mapping"],
                experiment["context"], experiment["space"],
            )
            state["sweep_evaluations"] += 1
        missing = []
    if missing:
        return None, missing
    first = cache[criterion.loss_names[0]]
    columns = [name for name in (
        "strength", "modified_min", "modified_max", "outside_reference_fraction",
    ) if name in first]
    combined = first[columns].copy()
    combined["total"] = 0.0
    errors = [[] for _ in range(len(combined))]
    for name, weight in zip(criterion.loss_names, criterion.weights):
        frame = cache[name]
        values = frame.get(f"term:{name}", pd.Series(np.nan, index=combined.index))
        combined[f"term:{name}"] = values
        combined[f"weighted:{name}"] = weight * values
        combined["total"] += weight * values
        for index, message in enumerate(frame["error"]):
            if message:
                errors[index].append(f"{name}: {message}")
    combined["error"] = ["; ".join(dict.fromkeys(messages)) for messages in errors]
    combined.loc[combined["error"] != "", "total"] = np.nan
    return combined, []

In [ ]:
def make_dashboard_controls(state):
    labels = [("MSE", "MSELoss"), ("MAE", "L1Loss"), ("Smooth L1", "SmoothL1Loss"),
              ("Huber", "HuberLoss"), ("SSIM", "SSIMLoss")]
    controls = {
        "split": widgets.Dropdown(options=["train", "val", "test"], value=SPLIT, description="Split"),
        "image": widgets.Dropdown(description="Image", layout=widgets.Layout(width="700px")),
        "previous": widgets.Button(description="Previous", layout=widgets.Layout(width="95px")),
        "next": widgets.Button(description="Next", layout=widgets.Layout(width="75px")),
        "mode": widgets.ToggleButtons(options=[("Single loss", "single"), ("Combination", "combination")],
                                     value="combination" if len(LOSS_NAMES) > 1 else "single"),
        "single_loss": widgets.ToggleButtons(options=labels, value=LOSS_NAMES[0]),
        "loss_checks": {}, "loss_weights": {},
        "modification": widgets.Dropdown(options=list(MODIFICATIONS), value=MODIFICATION, description="Change"),
        "strength": widgets.FloatSlider(description="Strength", continuous_update=False, readout_format=".3g",
                                        layout=widgets.Layout(width="650px")),
        "range_min": widgets.FloatText(value=float(strengths.min()), description="Sweep min"),
        "range_max": widgets.FloatText(value=float(strengths.max()), description="Sweep max"),
        "steps": widgets.BoundedIntText(value=N_STEPS, min=2, max=201, description="Points"),
        "normalize": widgets.Checkbox(value=NORMALIZATION["enabled"], description="Normalize loss inputs"),
        "transform": widgets.Dropdown(options=["linear", "log1p", "signed_log1p"],
                                      value=NORMALIZATION["transform"], description="Transform"),
        "bounds": widgets.ToggleButtons(options=[("Per image", "per_image"), ("Shared", "shared")],
                                        value=NORMALIZATION["bounds"]),
        "log_scale": widgets.FloatText(value=NORMALIZATION["log_scale"], description="Log scale"),
        "space": widgets.ToggleButtons(options=[("Transformed", "transformed"), ("Physical", "physical")],
                                       value=COMPARISON_SPACE),
        "stretch": widgets.ToggleButtons(options=["linear", "asinh"], value=DISPLAY_STRETCH, description="Display"),
        "gradient": widgets.Checkbox(value=SHOW_GRADIENT, description="Show gradient"),
        "table": widgets.Checkbox(value=False, description="Show sweep table"),
        "angle": widgets.FloatSlider(value=MODIFICATION_OPTIONS["shift_angle_deg"], min=-180, max=180,
                                     step=5, description="Angle [deg]", continuous_update=False),
        "fill": widgets.FloatText(value=MODIFICATION_OPTIONS["fill_value"], description="Fill [Jy/beam]"),
        "core_radius": widgets.BoundedFloatText(value=MODIFICATION_OPTIONS["core_radius_px"], min=0.1, max=IMAGE_SIZE, description="Core radius"),
        "core_edge": widgets.BoundedFloatText(value=MODIFICATION_OPTIONS["core_edge_px"], min=0.1, max=IMAGE_SIZE, description="Core edge"),
        "jet_sigma": widgets.FloatSlider(value=MODIFICATION_OPTIONS["jet_min_sigma"], min=0, max=10, step=0.25,
                                         description="Jet min RMS", continuous_update=False),
        "jet_fraction": widgets.FloatSlider(value=MODIFICATION_OPTIONS["jet_max_peak_fraction"], min=0.01, max=1,
                                            step=0.01, description="Jet max/peak", continuous_update=False),
        "artifact_x": widgets.FloatSlider(value=MODIFICATION_OPTIONS["artifact_center_fraction"][0], min=0, max=1,
                                          step=0.01, description="Artifact x", continuous_update=False),
        "artifact_y": widgets.FloatSlider(value=MODIFICATION_OPTIONS["artifact_center_fraction"][1], min=0, max=1,
                                          step=0.01, description="Artifact y", continuous_update=False),
        "artifact_width": widgets.BoundedFloatText(value=MODIFICATION_OPTIONS["artifact_sigma_px"], min=0.1, max=IMAGE_SIZE, description="Width [px]"),
        "noise_seed": widgets.BoundedIntText(value=SEED, min=0, max=2**31-1, description="Noise seed"),
        "run": widgets.Button(description="Run sweep", button_style="primary", icon="line-chart"),
        "reset": widgets.Button(description="Reset controls", icon="undo"),
        "status": widgets.HTML(), "objective": widgets.HTML(), "reference_info": widgets.HTML(),
        "preview_output": widgets.Output(), "curve_output": widgets.Output(),
    }
    rows = []
    defaults = dict(zip(LOSS_NAMES, LOSS_WEIGHTS))
    for label, name in labels:
        controls["loss_checks"][name] = widgets.Checkbox(value=name in LOSS_NAMES, description=label,
                                                        layout=widgets.Layout(width="140px"))
        controls["loss_weights"][name] = widgets.FloatSlider(
            value=defaults.get(name, 1.0), min=0, max=1, step=0.01, description="Weight",
            continuous_update=False, readout_format=".2f", layout=widgets.Layout(width="350px"),
        )
        rows.append(widgets.HBox([controls["loss_checks"][name], controls["loss_weights"][name]]))
    controls["combination_box"] = widgets.VBox(rows)
    controls["normalization_box"] = widgets.VBox([controls["transform"], controls["bounds"], controls["log_scale"]])
    controls["option_rows"] = {
        "angle": (controls["angle"], {"shift", "core_shift"}),
        "fill": (controls["fill"], {"shift"}),
        "core": (widgets.HBox([controls["core_radius"], controls["core_edge"]]), {"core_shift", "jet_attenuation"}),
        "jet": (widgets.VBox([controls["jet_sigma"], controls["jet_fraction"]]), {"jet_attenuation"}),
        "artifact": (widgets.VBox([controls["artifact_x"], controls["artifact_y"], controls["artifact_width"]]), {"artifact"}),
        "seed": (controls["noise_seed"], {"noise"}),
    }
    objective_box = widgets.VBox([controls["mode"], controls["single_loss"], controls["combination_box"], controls["objective"]])
    modification_box = widgets.VBox([
        controls["modification"], *[row for row, _ in controls["option_rows"].values()],
        widgets.HBox([controls["range_min"], controls["range_max"], controls["steps"]]),
    ])
    normalization_box = widgets.VBox([
        controls["space"], controls["normalize"], controls["normalization_box"],
        widgets.HTML("<small>Both images use fixed reference bounds. Display stretch is separate; the inverse is not applied.</small>"),
    ])
    tabs = widgets.Tab(children=[objective_box, modification_box, normalization_box])
    for index, title in enumerate(["Loss", "Modification", "Normalization"]):
        tabs.set_title(index, title)
    controls["widget"] = widgets.VBox([
        widgets.HBox([controls["split"], controls["previous"], controls["next"]]),
        controls["image"], controls["reference_info"], tabs, controls["strength"],
        widgets.HBox([controls["stretch"], controls["gradient"], controls["table"]]),
        widgets.HBox([controls["run"], controls["reset"]]), controls["status"],
        controls["curve_output"], controls["preview_output"],
    ])
    return controls

In [ ]:
def render_dashboard(experiment, results, controls, preview_value):
    with controls["curve_output"]:
        clear_output(wait=True)
        if results is None:
            print("No cached curve for this selection. Run sweep to evaluate the strength range.")
        else:
            fig = plot_response(results, experiment["criterion"], experiment["label"],
                                experiment["strength"], preview_value)
            display(fig)
            plt.close(fig)
            if controls["table"].value:
                display(results)
            failed = results.loc[results["error"] != "", ["strength", "error"]]
            if not failed.empty:
                display(failed)
    with controls["preview_output"]:
        clear_output(wait=True)
        show_preview(
            experiment["strength"], experiment["reference"], experiment["modification"],
            experiment["criterion"], experiment["mapping"], experiment["context"],
            experiment["space"], None, experiment["label"],
            f"{experiment['path'].name} — {experiment['name']}",
            controls["stretch"].value, controls["gradient"].value,
        )

In [ ]:
def refresh_dashboard(controls, state, run_sweep=False):
    if state["suspend"]:
        return
    state["last_error"] = ""
    try:
        experiment = dashboard_experiment(controls, state)
        criterion = experiment["criterion"]
        controls["objective"].value = "<b>Effective weights:</b> " + ", ".join(
            f"{escape(name)} = {weight:.3f}" for name, weight in zip(criterion.loss_names, criterion.weights)
        )
        context = experiment["context"]
        controls["reference_info"].value = (
            f"<b>{escape(experiment['path'].name)}</b> · "
            f"RMS {context['noise_rms']:.4g} Jy/beam ({escape(context['noise_source'])}) · "
            f"jet mask: {int((context['jet_mask'] > 0).sum())} pixels"
        )
        preview_value = None
        preview_error = ""
        try:
            modified = experiment["modification"](experiment["reference"], experiment["strength"], context)
            total, _ = evaluate_loss(
                experiment["reference"], modified, criterion, experiment["mapping"], context, experiment["space"],
            )
            preview_value = float(total.detach())
        except (ValueError, RuntimeError, FloatingPointError, OverflowError) as error:
            preview_error = str(error)
        if run_sweep:
            controls["status"].value = "<span style='color:#555'>Calculating missing loss terms…</span>"
        results, missing = cached_dashboard_sweep(experiment, state, run_sweep)
        state.update(last_experiment=experiment, last_results=results, last_preview_value=preview_value)
        if preview_error:
            state["last_error"] = preview_error
            controls["status"].value = f"<span style='color:#b33'>Preview: {escape(preview_error)}</span>"
        else:
            curve_status = "Curve cached." if not missing else "Run sweep for: " + ", ".join(missing) + "."
            controls["status"].value = f"<b>Preview loss: {preview_value:.6g}</b> · {escape(curve_status)}"
        render_dashboard(experiment, results, controls, preview_value)
    except (ValueError, IndexError, FileNotFoundError, RuntimeError, FloatingPointError, OverflowError) as error:
        state["last_error"] = str(error)
        state.update(last_results=None, last_preview_value=None)
        controls["status"].value = f"<span style='color:#b33'>{escape(str(error))}</span>"
        controls["objective"].value = "<b>Selection requires correction.</b>"
        controls["curve_output"].clear_output()
        controls["preview_output"].clear_output()

In [ ]:
def build_loss_dashboard():
    initial_index = BATCH_INDEX * BATCH_SIZE + IMAGE_INDEX
    state = {
        "suspend": True, "loaders": {SPLIT: (data, loader)},
        "images": {(SPLIT, initial_index): (reference, reference_metadata, metadata_validity, reference_path)},
        "shared_bounds": dict(data.dataset_metadata), "contexts": {}, "curves": {},
        "sweep_evaluations": 0, "last_error": "", "last_results": None, "last_preview_value": None,
    }
    controls = make_dashboard_controls(state)

    def image_options(preferred=0):
        _, current_loader = dashboard_loader(state, controls["split"].value)
        controls["image"].options = [
            (f"{index + 1}: {Path(path).name}", index)
            for index, path in enumerate(current_loader.dataset.paths)
        ]
        controls["image"].value = min(int(preferred), len(current_loader.dataset) - 1)

    def synchronize():
        combination = controls["mode"].value == "combination"
        controls["single_loss"].layout.display = "none" if combination else ""
        controls["combination_box"].layout.display = "" if combination else "none"
        for name, weight in controls["loss_weights"].items():
            weight.disabled = not controls["loss_checks"][name].value
        controls["normalization_box"].layout.display = "" if controls["normalize"].value else "none"
        controls["log_scale"].layout.display = "" if controls["transform"].value != "linear" else "none"
        for row, modifications in controls["option_rows"].values():
            row.layout.display = "" if controls["modification"].value in modifications else "none"
        count = len(controls["image"].options)
        controls["previous"].disabled = controls["image"].value == 0
        controls["next"].disabled = controls["image"].value == count - 1
        lower, upper = controls["range_min"].value, controls["range_max"].value
        if math.isfinite(lower) and math.isfinite(upper) and lower < upper:
            slider = controls["strength"]
            slider.min, slider.max = min(slider.min, lower), max(slider.max, upper)
            slider.value = float(np.clip(slider.value, lower, upper))
            slider.min, slider.max = lower, upper
            slider.step = (upper - lower) / 100

    def changed(change):
        if state["suspend"]:
            return
        state["suspend"] = True
        try:
            if change["owner"] is controls["split"]:
                image_options()
            if change["owner"] is controls["modification"]:
                lower, upper = MODIFICATIONS[controls["modification"].value]["limits"]
                controls["range_min"].value, controls["range_max"].value = lower, upper
                synchronize()
                controls["strength"].value = (lower + upper) / 2
            synchronize()
        except (ValueError, IndexError, FileNotFoundError, RuntimeError) as error:
            state["last_error"] = str(error)
            controls["status"].value = f"<span style='color:#b33'>{escape(str(error))}</span>"
            return
        finally:
            state["suspend"] = False
        refresh_dashboard(controls, state)

    image_options(initial_index)
    synchronize()
    controls["strength"].value = float(np.clip(preview_strength, controls["strength"].min, controls["strength"].max))
    observed_keys = [
        "split", "image", "mode", "single_loss", "modification", "strength",
        "range_min", "range_max", "steps", "normalize", "transform", "bounds",
        "log_scale", "space", "stretch", "gradient", "table", "angle", "fill",
        "core_radius", "core_edge", "jet_sigma", "jet_fraction",
        "artifact_x", "artifact_y", "artifact_width", "noise_seed",
    ]
    defaults = {name: controls[name].value for name in observed_keys}
    check_defaults = {name: widget.value for name, widget in controls["loss_checks"].items()}
    weight_defaults = {name: widget.value for name, widget in controls["loss_weights"].items()}

    def reset(_):
        state["suspend"] = True
        try:
            controls["split"].value = defaults["split"]
            image_options(defaults["image"])
            for name in observed_keys:
                if name not in {"split", "image", "strength"}:
                    controls[name].value = defaults[name]
            for name, value in check_defaults.items():
                controls["loss_checks"][name].value = value
            for name, value in weight_defaults.items():
                controls["loss_weights"][name].value = value
            synchronize()
            controls["strength"].value = defaults["strength"]
        finally:
            state["suspend"] = False
        refresh_dashboard(controls, state)

    for name in observed_keys:
        controls[name].observe(changed, names="value")
    for widget in [*controls["loss_checks"].values(), *controls["loss_weights"].values()]:
        widget.observe(changed, names="value")
    controls["previous"].on_click(lambda _: setattr(controls["image"], "value", max(0, controls["image"].value - 1)))
    controls["next"].on_click(lambda _: setattr(controls["image"], "value", min(len(controls["image"].options) - 1, controls["image"].value + 1)))
    controls["run"].on_click(lambda _: refresh_dashboard(controls, state, run_sweep=True))
    controls["reset"].on_click(reset)
    state["suspend"] = False
    return {
        "controls": controls, "state": state, "widget": controls["widget"],
        "refresh": partial(refresh_dashboard, controls, state),
    }

## Activate the panel

The controls and plots appear in the panel near the top of the notebook. A static experiment is retained for kernels without widgets. Failed comparisons are reported explicitly rather than reusing an old curve.

In [ ]:
if WIDGETS_AVAILABLE:
    dashboard = build_loss_dashboard()
    dashboard_mount.children = (dashboard["widget"],)
    dashboard["refresh"]()
else:
    results = run_sweep(
        reference, selected_modification["function"], strengths,
        loss_function, intensity_transform, context, COMPARISON_SPACE,
    )
    identity = results.loc[results["strength"] == 0]
    if identity.empty or identity["error"].iloc[0]:
        raise RuntimeError(f"The unchanged-image control failed: {identity.to_dict('records')}")
    if not np.isclose(identity["total"].iloc[0], 0.0, atol=1e-5):
        raise RuntimeError("The unchanged-image control did not produce approximately zero loss.")
    display(results)
    failed = results.loc[results["error"] != "", ["strength", "error"]]
    if not failed.empty:
        print(f"{len(failed)} comparison point(s) were invalid:")
        display(failed)
    
    preview_callback = partial(
        show_preview,
        reference=reference,
        modification=selected_modification["function"],
        criterion=loss_function,
        mapping=intensity_transform,
        context=context,
        space=COMPARISON_SPACE,
        results=results,
        axis_label=selected_modification["label"],
        title=f"{reference_path.name} — {MODIFICATION}",
        stretch=DISPLAY_STRETCH,
        show_gradient=SHOW_GRADIENT,
    )
    preview_callback(preview_strength)

## Interpretation

The response curve describes the selected objective under the chosen modification, image, and normalization. Its magnitude is meaningful within that experiment; it is not a ranking of unrelated losses. The gradient maps describe a local image update, not a change to trained model parameters.

For mixtures, weighted contributions expose which term drives the current objective. All regression and SSIM terms use the same selected image pair. Comparing a diffusion model's actual noise-regression objective would be a separate checkpoint experiment.

The image-space jet mask and artifact model are controlled examples. Aperture brightness sums in Jy/beam are not automatically integrated flux densities in Jy, and full-image metadata does not automatically describe the central crop.

Changing an objective updates its preview immediately. **Run sweep** calculates missing terms for the active image, modification, normalization, and strength interval. Reusing those terms keeps weight adjustments fast. A missing curve is labelled explicitly; a previous experiment's curve is not shown as current.